# 🧪 Taller - Gestos con Cámara Web: Control Visual con MediaPipe

Usar la webcam y la biblioteca MediaPipe para detectar gestos de manos y ejecutar acciones visuales en tiempo real. El propósito es explorar cómo las interfaces naturales pueden usarse para interactuar con la pantalla de forma intuitiva, sin hardware adicional.

### Importar librerias

In [42]:
import cv2, math
import mediapipe as mp

## Captura en tiempo real de cámara web

Se genero la función que captura video en una ventana externa. Para terminar su ejecución se debe presionar la letra 'q'.

In [43]:
def capture_webcam_feed():
    """
    Activa la cámara web y muestra el feed de video en tiempo real.
    Presiona 'q' para salir.
    """
    # Inicializa la captura de video desde la primera cámara disponible (índice 0)
    cap = cv2.VideoCapture(0)

    # Verifica si la cámara se abrió correctamente
    if not cap.isOpened():
        print("Error: No se pudo abrir la cámara web.")
        return

    while True:
        # Lee un frame de la cámara
        ret, frame = cap.read()

        # Si no se pudo leer el frame (ej. fin del stream)
        if not ret:
            break

        # Muestra el frame en una ventana llamada 'Webcam Feed'
        cv2.imshow('Webcam Feed', frame)

        # Espera 1 milisegundo por una tecla.
        # Si la tecla presionada es 'q' (código ASCII 113), sale del bucle.
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Libera el objeto de captura de video
    cap.release()
    # Destruye todas las ventanas de OpenCV
    cv2.destroyAllWindows()

Ejecutar la siguiente celda para capturar video con la webcam

In [44]:
capture_webcam_feed()

## Detectar manos usando MediaPipe Hands

Utiliza la misma logica de antes para capturar cámara pero ahora agregamos la lógica para la detección manos.

In [35]:
def hand_detection_webcam():
    """
    Activa la cámara web, detecta manos usando MediaPipe Hands
    y dibuja los landmarks y conexiones.
    Presiona 'q' para salir.
    """
    # Inicializa la captura de video
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        return

    # Inicializa el módulo de MediaPipe Hands
    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,  # Detecta hasta 2 manos
        min_detection_confidence=0.6,
        min_tracking_confidence=0.6
    )

    # Inicializa las utilidades de dibujo de MediaPipe
    mp_drawing = mp.solutions.drawing_utils
    mp_drawing_styles = mp.solutions.drawing_styles

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Voltea la imagen horizontalmente para una vista tipo "selfie"
        frame = cv2.flip(frame, 1)

        # Convierte la imagen BGR de OpenCV a RGB, que es el formato que espera MediaPipe
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Procesa la imagen y detecta las manos
        results = hands.process(rgb_frame)

        # Dibuja los landmarks de las manos si se detectan
        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                # Dibuja los puntos (landmarks) de la mano
                mp_drawing.draw_landmarks(
                    frame,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS,
                    mp_drawing_styles.get_default_hand_landmarks_style(),
                    mp_drawing_styles.get_default_hand_connections_style()
                )

        # Muestra el frame resultante
        cv2.imshow('Hand Detection with MediaPipe', frame)

        # Salir con 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Libera los recursos
    cap.release()
    cv2.destroyAllWindows()
    hands.close() # Es buena práctica liberar el recurso de MediaPipe

In [36]:
hand_detection_webcam()

## Cálculo de métricas importantes

In [38]:
def count_extended_fingers(hand_landmarks_obj, hand_label, image_width, image_height):
    """
    Cuenta el número de dedos extendidos para una mano detectada.
    Basado en la posición de los landmarks de las puntas de los dedos y las articulaciones.

    Args:
        hand_landmarks_obj (mediapipe.framework.formats.landmark_pb2.NormalizedLandmarkList):
            Objeto que contiene los landmarks de la mano.
        hand_label (str): Etiqueta de la mano ('Left' o 'Right') para ajustar la lógica del pulgar.
        image_width (int): Ancho de la imagen para convertir coordenadas normalizadas a píxeles.
        image_height (int): Alto de la imagen para convertir coordenadas normalizadas a píxeles.

    Returns:
        int: Número de dedos extendidos.
    """
    landmarks = hand_landmarks_obj.landmark
    extended_fingers = 0

    # Puntos de referencia para cada dedo:
    # Pulgar: tip (4), IP (3)
    # Índice: tip (8), PIP (6)
    # Medio: tip (12), PIP (10)
    # Anular: tip (16), PIP (14)
    # Meñique: tip (20), PIP (18)
    
    mp_hands = mp.solutions.hands

    # Coordenadas en píxeles (convertimos de normalizadas [0,1] a píxeles)
    finger_tips_y = [landmarks[mp_hands.HandLandmark.INDEX_FINGER_TIP].y * image_height,
                     landmarks[mp_hands.HandLandmark.MIDDLE_FINGER_TIP].y * image_height,
                     landmarks[mp_hands.HandLandmark.RING_FINGER_TIP].y * image_height,
                     landmarks[mp_hands.HandLandmark.PINKY_TIP].y * image_height]

    finger_pips_y = [landmarks[mp_hands.HandLandmark.INDEX_FINGER_PIP].y * image_height,
                     landmarks[mp_hands.HandLandmark.MIDDLE_FINGER_PIP].y * image_height,
                     landmarks[mp_hands.HandLandmark.RING_FINGER_PIP].y * image_height,
                     landmarks[mp_hands.HandLandmark.PINKY_PIP].y * image_height]

    # Lógica para los 4 dedos (índice, medio, anular, meñique)
    # Un dedo está extendido si la punta está por encima de la articulación PIP
    for i in range(4):
        if finger_tips_y[i] < finger_pips_y[i]:
            extended_fingers += 1

    # Lógica para el pulgar
    # El pulgar es un poco más complejo debido a su movimiento lateral.
    thumb_tip_x = landmarks[mp_hands.HandLandmark.THUMB_TIP].x
    thumb_mcp_x = landmarks[mp_hands.HandLandmark.THUMB_MCP].x

    if hand_label == 'Right':
        # Para mano derecha (vista de espejo), si la punta del pulgar está a la izquierda de la base, está extendido.
        # Es decir, THUMB_TIP.x < THUMB_MCP.x
        if thumb_tip_x < thumb_mcp_x:
            extended_fingers += 1
    elif hand_label == 'Left':
        # Para mano izquierda (vista de espejo), si la punta del pulgar está a la derecha de la base, está extendido.
        # Es decir, THUMB_TIP.x > THUMB_MCP.x
        if thumb_tip_x > thumb_mcp_x:
            extended_fingers += 1

    # Consideración alternativa para el pulgar (más robusta):
    # Compara la distancia entre la punta del pulgar y el nudillo del índice.
    # Si la punta del pulgar está muy separada, es probable que esté extendido.
    # thumb_tip = (landmarks[mp_hands.HandLandmark.THUMB_TIP].x, landmarks[mp_hands.HandLandmark.THUMB_TIP].y)
    # index_mcp = (landmarks[mp_hands.HandLandmark.INDEX_FINGER_MCP].x, landmarks[mp_hands.HandLandmark.INDEX_FINGER_MCP].y)
    # distance = math.hypot((thumb_tip[0] - index_mcp[0]) * image_width, (thumb_tip[1] - index_mcp[1]) * image_height)
    # if distance > some_threshold: # Puedes ajustar este umbral
    #     extended_fingers += 1


    return extended_fingers

def calculate_distance(p1, p2, image_width, image_height):
    """
    Calcula la distancia euclidiana entre dos puntos de referencia.
    Los puntos son coordenadas normalizadas [0,1], se convierten a píxeles.
    """
    x1, y1 = p1.x * image_width, p1.y * image_height
    x2, y2 = p2.x * image_width, p2.y * image_height
    return math.hypot(x2 - x1, y2 - y1)

In [39]:
def process_hand_landmarks(image, hands_model):
    """
    Procesa una imagen para detectar manos usando MediaPipe Hands,
    dibuja los landmarks y calcula métricas útiles.

    Args:
        image (numpy.ndarray): El frame de la cámara en formato BGR.
        hands_model (mediapipe.python.solutions.hands.Hands): Instancia del modelo MediaPipe Hands.

    Returns:
        numpy.ndarray: La imagen con los landmarks de las manos y métricas dibujadas.
        dict: Un diccionario con las métricas calculadas para cada mano.
    """
    image_height, image_width, _ = image.shape
    
    mp_hands = mp.solutions.hands
    
    # Inicializa las utilidades de dibujo de MediaPipe
    mp_drawing = mp.solutions.drawing_utils
    mp_drawing_styles = mp.solutions.drawing_styles

    # Voltea la imagen horizontalmente para una vista tipo "selfie"
    image = cv2.flip(image, 1)

    # Convierte la imagen BGR de OpenCV a RGB, que es el formato que espera MediaPipe
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Procesa la imagen y detecta las manos
    results = hands_model.process(rgb_image)

    hand_metrics = {
        'num_hands_detected': 0,
        'hand_details': []
    }

    # Dibuja los landmarks de las manos si se detectan
    if results.multi_hand_landmarks:
        hand_metrics['num_hands_detected'] = len(results.multi_hand_landmarks)

        for hand_idx, hand_landmarks in enumerate(results.multi_hand_landmarks):
            # Obtener el tipo de mano (izquierda/derecha)
            # MediaPipe devuelve una lista de clasificación, accedemos al primer elemento
            hand_label = results.multi_handedness[hand_idx].classification[0].label

            # --- Métrica 1: Número de dedos extendidos ---
            extended_fingers = count_extended_fingers(hand_landmarks, hand_label, image_width, image_height)

            # --- Métrica 2: Distancia entre pulgar e índice ---
            thumb_tip = hand_landmarks.landmark[mp_hands.HandLandmark.THUMB_TIP]
            index_tip = hand_landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP]
            thumb_index_distance = calculate_distance(thumb_tip, index_tip, image_width, image_height)


            hand_details = {
                'label': hand_label,
                'extended_fingers': extended_fingers,
                'thumb_index_distance': f"{thumb_index_distance:.2f} px",
            }
            hand_metrics['hand_details'].append(hand_details)

            # Dibuja los puntos (landmarks) de la mano
            mp_drawing.draw_landmarks(
                image,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style()
            )

            # Muestra las métricas en la imagen
            text_x_offset = 10
            text_y_offset = 30 + (hand_idx * 150) # Espacio para cada mano
            cv2.putText(image, f"Mano: {hand_label}", (text_x_offset, text_y_offset),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2, cv2.LINE_AA)
            cv2.putText(image, f"Dedos Extendidos: {extended_fingers}", (text_x_offset, text_y_offset + 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2, cv2.LINE_AA)
            cv2.putText(image, f"Distancia Pulgar-Indice: {thumb_index_distance:.2f} px", (text_x_offset, text_y_offset + 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2, cv2.LINE_AA)


    # Mostrar el número total de manos detectadas
    cv2.putText(image, f"Manos Detectadas: {hand_metrics['num_hands_detected']}", (image_width - 350, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2, cv2.LINE_AA)

    return image, hand_metrics

In [40]:
def capture_webcam_feed_with_hands_metrics():
    """
    Activa la cámara web, detecta manos utilizando MediaPipe Hands,
    calcula y muestra métricas en tiempo real.
    Presiona 'q' para salir.
    """
    mp_hands = mp.solutions.hands
    
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Error: No se pudo abrir la cámara web.")
        return

    hands = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.75,
        min_tracking_confidence=0.75
    )

    print("Cámara web con detección de manos y métricas activada. Presiona 'q' para salir.")

    while True:
        ret, frame = cap.read()

        if not ret:
            print("Error: No se pudo recibir el frame. Saliendo...")
            break

        frame_with_data, hand_metrics = process_hand_landmarks(frame, hands)


        cv2.imshow('Webcam Feed with Hand Detection & Metrics', frame_with_data)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    hands.close()

In [45]:
capture_webcam_feed_with_hands_metrics()

Cámara web con detección de manos y métricas activada. Presiona 'q' para salir.
